# 01.03 VectorAdd 算子实验

## 小节概述

本节是一个引导式实验。你将沿着 CANN Learning Hub Vector 算子开发的典型路径，依次完成算子分析、Tiling 推导、Kernel 分阶段阅读、Host 调用、编译运行和结果解释。实验使用的计算公式为 <code>z[i] = x[i] + y[i]</code>。

### 学习前置要求

- 已完成 [01.02 环境与工程一键启动](01.02_environment_and_project.ipynb)；
- 理解一维连续数组、整数除法与左闭右开区间；
- 已进入带有 CANN 与昇腾 NPU 的 CANNLab 环境。

### 本节目标

完成本节后，你应当能够：

- 说明一维数组如何切分到多个 Block 和每核多个 Tile；
- 按 <code>Init → Process → CopyIn/Compute/CopyOut</code> 的顺序读懂 Ascend C Kernel；
- 说明 <code>TPipe</code>、<code>TQue</code>、<code>GlobalTensor</code> 和 <code>LocalTensor</code> 的作用；
- 完成单缓冲 VectorAdd 的编译、运行和精度检查；
- 根据 Host 报错或 <code>METRIC</code> 输出定位常见问题。

<strong>建议用时：</strong> 60 分钟。<strong>本节产出：</strong> 一次合法 Shape 的运行记录、三类非法参数的观察记录，以及课后实践中的参数推导。

### 章节内容

1. 分析 VectorAdd 规格与核内数据通路；
2. 按 <code>Init/Process/CopyIn/Compute/CopyOut</code> 阅读 Kernel；
3. 理解 Host 启动、编译运行与结果校验；
4. 观察非法参数并完成巩固练习。


## 教程内容

### 1. 准备实验目录

先运行下面的单元，定位课程目录、演示源码和临时构建目录。本节只读取 <code>src/demo</code> 中已经完成的参考工程；需要独立补全 Device 代码的综合任务在 01.05 中进行。

<strong>检查点：</strong> 输出的三个路径均存在，并且 <code>vector_add.asc</code> 可以被找到。若提示不在仓库中，请从 <code>cann-learning-hub</code> 仓库内重新打开本 Notebook。


In [ ]:
from pathlib import Path
import getpass
import os
import shutil
import subprocess
import sys
import tempfile

previous_repo = globals().get('REPO_ROOT')
try:
    start = Path.cwd().resolve()
except FileNotFoundError:
    cached_repo = Path(previous_repo) if previous_repo is not None else None
    if cached_repo is None or not (cached_repo / 'contrib/tutorials/data_structures_compute').is_dir():
        raise RuntimeError(
            '当前内核的工作目录已被清理，请重启内核后从本节第一个代码单元开始运行'
        ) from None
    os.chdir(cached_repo)
    start = cached_repo.resolve()
search_roots = []
if previous_repo is not None:
    search_roots.append(Path(previous_repo).resolve())
search_roots.extend([start, *start.parents])
REPO_ROOT = next(
    (p for p in search_roots if (p / 'contrib/tutorials/data_structures_compute').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('请从 cann-learning-hub 仓库内打开本 Notebook')
os.chdir(REPO_ROOT)

COURSE = REPO_ROOT / 'contrib/tutorials/data_structures_compute'
CHAPTER = COURSE / '01_basic_operations'
SOURCE = CHAPTER / 'src/demo/vector_add.asc'
CMAKE_FILE = CHAPTER / 'src/demo/CMakeLists.txt'
USER_KEY = f'uid_{os.getuid()}' if hasattr(os, 'getuid') else f'user_{getpass.getuser()}'
USER_TEMP_ROOT = Path(tempfile.gettempdir()) / f'cannlab_data_structures_compute_{USER_KEY}'
WORK = USER_TEMP_ROOT / '01_vector_add_demo'

def show_between(path, start_marker, end_marker):
    lines = path.read_text(encoding='utf-8').splitlines()
    start_index = next((i for i, line in enumerate(lines) if start_marker in line), None)
    if start_index is None:
        raise ValueError(f'未找到起始标记: {start_marker}')
    end_index = next(
        (i for i in range(start_index + 1, len(lines)) if end_marker in lines[i]),
        None,
    )
    if end_index is None:
        raise ValueError(f'未找到结束标记: {end_marker}')
    for index in range(start_index, end_index):
        print(f'{index + 1:>3}: {lines[index]}')

def show_matches(path, patterns):
    lines = path.read_text(encoding='utf-8').splitlines()
    for pattern in patterns:
        matches = [(i, line) for i, line in enumerate(lines) if pattern in line]
        if not matches:
            raise ValueError(f'未找到代码: {pattern}')
        for index, line in matches:
            print(f'{index + 1:>3}: {line}')

print('仓库目录 :', REPO_ROOT)
print('演示源码 :', SOURCE)
print('构建目录 :', WORK)
assert SOURCE.is_file() and CMAKE_FILE.is_file()


### 2. 分析算子规格

先明确算子要完成什么，再讨论如何实现。

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>项目</th><th style='text-align: left;'>本实验规格</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'>输入</td><td style='text-align: left;'>一维连续数组 <code>x</code>、<code>y</code></td></tr>
    <tr><td style='text-align: left;'>输出</td><td style='text-align: left;'>一维连续数组 <code>z</code></td></tr>
    <tr><td style='text-align: left;'>数据类型</td><td style='text-align: left;'><code>float32</code></td></tr>
    <tr><td style='text-align: left;'>逻辑 Shape</td><td style='text-align: left;'><code>[N]</code>，三个数组长度相同</td></tr>
    <tr><td style='text-align: left;'>计算公式</td><td style='text-align: left;'>对所有 <code>0 ≤ i &lt; N</code>，<code>z[i] = x[i] + y[i]</code></td></tr>
    <tr><td style='text-align: left;'>默认参数</td><td style='text-align: left;'><code>N=16384</code>、<code>blockDim=8</code>、<code>tileCount=8</code></td></tr>
    <tr><td style='text-align: left;'>精度判据</td><td style='text-align: left;'>所有输出均为有限数，且 <code>max_abs_error ≤ 1e-6</code></td></tr>
  </tbody>
</table>

本实验只接收可以等分且满足搬运对齐要求的参数。Host 会在发射 Kernel 前检查：

1. <code>N % blockDim == 0</code>；
2. <code>blockLength % tileCount == 0</code>；
3. <code>tileLength × sizeof(float) % 32 == 0</code>。

<strong>检查点：</strong> 能够用一句话说明三个条件分别保证“分核完整”“核内分 Tile 完整”和“DataCopy 长度按 32 Byte 对齐”。


### 3. 理解数据通路

本实验沿 Vector 算子课程的 SIMD Membase 教学路径展开。<strong>SIMD</strong>（Single Instruction, Multiple Data，单指令多数据）表示一条 Vector 指令同时处理一组元素，例如 <code>AscendC::Add</code> 一次完成整个 Tile 的逐元素相加；<strong>Membase</strong> 表示通过 <code>LocalTensor</code> 和 Local Memory 组织搬运与计算，而不是由学习者直接编写寄存器操作。

在本算子中，数据从 GM 搬入 Local Memory，由 UB 承载 <code>TQue</code> 队列槽位；Vector 指令完成计算后，结果再从 UB 写回 GM。

![VectorAdd 从 GM 到 Local Memory 再写回 GM 的数据通路](./images/vector_add_dataflow.svg)

<p><strong>图 1</strong> 单个 Tile 沿 SIMD Membase 路径完成搬入、计算和搬出。</p>

每个 Block 只处理自己负责的一段 GM。进入核内后，一个 Tile 依次经过三个阶段：

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>阶段</th><th style='text-align: left;'>数据移动或计算</th><th style='text-align: left;'>队列生命周期</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'>CopyIn</td><td style='text-align: left;'><code>x/y: GM → Local Memory</code></td><td style='text-align: left;'><code>AllocTensor → DataCopy → EnQue</code></td></tr>
    <tr><td style='text-align: left;'>Compute</td><td style='text-align: left;'><code>z = x + y</code></td><td style='text-align: left;'><code>DeQue(x/y) → AllocTensor(z) → Add → EnQue(z) → FreeTensor(x/y)</code></td></tr>
    <tr><td style='text-align: left;'>CopyOut</td><td style='text-align: left;'><code>z: Local Memory → GM</code></td><td style='text-align: left;'><code>DeQue(z) → DataCopy → FreeTensor(z)</code></td></tr>
  </tbody>
</table>

<code>GlobalTensor</code> 是当前核所负责 GM 区间的视图，<code>LocalTensor</code> 是核内缓冲的视图；<code>TPipe</code> 负责初始化流水资源，<code>TQue</code> 通过入队和出队表达阶段间依赖。输入队列位于 <code>VECIN</code>，输出队列位于 <code>VECOUT</code>。

> 本实验 01 默认使用单缓冲，因此每个队列只有 1 个槽位。<code>tileCount</code> 表示每核需要访问的真实 GM Tile 数，与队列槽位数是两个独立概念。改变槽位数不能改变数据覆盖范围。


### 4. 推导 Tiling 参数

![一维数组的 Block 与 Tile 切分](./images/tiling_map.svg)

<p><strong>图 2</strong> 默认参数下的两级切分，并突出 Block 2、Tile 3 的全局区间。</p>

本实验采用等长切分。Block 和 Tile 编号均从 0 开始：

<pre><code>blockLength = N / blockDim
tileLength  = blockLength / tileCount
blockOffset = blockIdx × blockLength
tileOffset  = tileIdx × tileLength
globalStart = blockOffset + tileOffset</code></pre>

运行下面的单元，复核默认参数，并跟踪 Block 2 的 Tile 3。


In [ ]:
N = 16384
block_dim = 8
tile_count = 8
block_idx = 2
tile_idx = 3
element_bytes = 4

block_length = N // block_dim
tile_length = block_length // tile_count
block_offset = block_idx * block_length
tile_offset = tile_idx * tile_length
global_start = block_offset + tile_offset
global_end = global_start + tile_length
tile_bytes = tile_length * element_bytes

assert N % block_dim == 0
assert block_length % tile_count == 0
assert tile_bytes % 32 == 0
print(f'blockLength = {block_length}')
print(f'tileLength  = {tile_length}')
print(f'tileBytes   = {tile_bytes}')
print(f'Block {block_idx}, Tile {tile_idx}: [{global_start}, {global_end})')


<strong>预期结果：</strong> <code>blockLength=2048</code>、<code>tileLength=256</code>、<code>tileBytes=1024</code>，Block 2 的 Tile 3 对应全局区间 <code>[4864, 5120)</code>。

演示工程只需把总长度和每核 Tile 数传给 Device。<code>blockDim</code> 是 Kernel 发射配置，Device 通过 <code>GetBlockNum()</code> 和 <code>GetBlockIdx()</code> 获取分核信息。运行下一单元查看 Tiling 数据结构。


In [ ]:
show_between(
    SOURCE,
    'struct VectorAddTilingData',
    '// ---------------- AI Core kernel',
)


<strong>完成检查：</strong>

- <code>totalLength</code> 决定全局数据总量；
- <code>tileCount</code> 是每核真实 GM Tile 数，默认值为 8；
- <code>TQue</code> 的 queue depth 固定为 <code>kQueueDepth=1</code>；<code>InitBuffer</code> 分配的物理缓冲数由 <code>kBufferNum=1</code> 决定。二者都不写入 Tiling 数据，也不参与 <code>tileLength</code> 的计算。


### 5. 分步实现 Kernel

接下来按照执行关系阅读 <code>KernelVectorAdd</code>。每一步都包含“输入是什么、进行了什么操作、队列状态如何变化”三个观察点。不要一次性浏览整份源码。

#### 5.1 Init：绑定每核 GM 区间并初始化队列

<code>Init</code> 接收三个 GM 地址以及 Tiling 参数，完成三项工作：计算当前核的切分、把 GM 视图平移到当前核起点、为三个队列分配 Local Memory 槽位。运行下面的单元查看实现。


In [ ]:
show_between(
    SOURCE,
    '    __aicore__ inline void Init(',
    '    __aicore__ inline void Process()',
)
print()
print('队列与流水对象：')
show_matches(
    SOURCE,
    [
        'AscendC::TPipe pipe_',
        'AscendC::TQue<AscendC::TPosition::VECIN',
        'AscendC::TQue<AscendC::TPosition::VECOUT',
    ],
)


对照源码完成以下检查：

1. <code>blockOffset = blockLength × GetBlockIdx()</code>，不同 Block 因而绑定互不重叠的 GM 区间；
2. <code>SetGlobalBuffer</code> 中的地址偏移以 <code>float</code> 元素为单位，长度也是元素数；
3. 每个队列槽位占 <code>tileLength × sizeof(float)</code> Byte；默认配置下每个槽位为 1024 Byte；
4. 对单个逻辑 Block（每核实例），单缓冲共有两个输入槽位和一个输出槽位，因此默认队列缓冲总量为 <code>3 × 1 × 1024 = 3072</code> Byte。

> <code>InitBuffer</code> 的第二个参数是队列槽位数，不是 Tile 数。即使以后把槽位改为 2，<code>tileCount_</code> 仍然表示需要处理的 8 个 GM Tile。


#### 5.2 Process：按真实 Tile 数驱动循环

<code>Process</code> 决定每个核执行多少次三阶段处理。运行下一单元，观察循环边界和三个阶段的调用顺序。


In [ ]:
show_between(
    SOURCE,
    '    __aicore__ inline void Process()',
    'private:',
)


<strong>观察结果：</strong> <code>loopCount</code> 直接等于 <code>tileCount_</code>。默认情况下每个核循环 8 次，每次覆盖 256 个元素，因此正好覆盖每核的 2048 个元素。

当前单缓冲实现按 <code>CopyIn → Compute → CopyOut</code> 顺序处理每个 Tile。循环次数不能乘以或除以缓冲槽位数，否则会造成重复计算或数据遗漏。


#### 5.3 CopyIn：从 GM 搬入一个 Tile

<code>progress</code> 是当前核内的 Tile 编号。<code>CopyIn</code> 先取得空闲输入槽位，再从当前核的 GM 视图中搬运一个 Tile，最后把 LocalTensor 入队交给 Compute。


In [ ]:
show_between(
    SOURCE,
    '    __aicore__ inline void CopyIn',
    '    __aicore__ inline void Compute()',
)


以 Block 2、Tile 3 为例：<code>xGm_</code> 和 <code>yGm_</code> 已在 Init 中平移到全局下标 4096；CopyIn 再使用核内偏移 <code>3 × 256 = 768</code>，最终读取全局区间 <code>[4864, 5120)</code>。

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>时刻</th><th style='text-align: left;'>输入队列状态</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>AllocTensor</code> 后</td><td style='text-align: left;'>获得可写的空闲槽位</td></tr>
    <tr><td style='text-align: left;'><code>DataCopy</code> 后</td><td style='text-align: left;'>槽位中保存当前 Tile 数据</td></tr>
    <tr><td style='text-align: left;'><code>EnQue</code> 后</td><td style='text-align: left;'>数据对 Compute 可见</td></tr>
  </tbody>
</table>

<strong>检查点：</strong> <code>offset</code> 与 <code>tileLength_</code> 都是元素数；默认 <code>float32</code> 搬运长度为 1024 Byte，满足 32 Byte 对齐。


#### 5.4 Compute：调用 Vector Add

Compute 从两个输入队列取出同一 Tile，为输出申请槽位，并调用 Ascend C Vector API 完成逐元素加法。运行下面的单元，重点观察 <code>DeQue</code>、<code>Add</code>、<code>EnQue</code> 和 <code>FreeTensor</code> 的先后关系。


In [ ]:
show_between(
    SOURCE,
    '    __aicore__ inline void Compute()',
    '    __aicore__ inline void CopyOut',
)


<code>AscendC::Add(zLocal, xLocal, yLocal, tileLength_)</code> 的最后一个参数是参与计算的元素数。默认一次处理 256 个 <code>float</code> 元素。

输出必须先 <code>EnQue</code>，CopyOut 才能取得它；输入完成计算后再 <code>FreeTensor</code>，槽位才可以被后续 Tile 复用。若遗漏释放，单缓冲队列很快会没有可分配槽位。

源码中的 <code>STUDENT_MODE</code> 和 <code>kComputeRepeat</code> 用于 01.05 的练习及后续性能实验。它们不改变本节默认路径：<code>kComputeRepeat=1</code>，直接调用一次 <code>AscendC::Add</code>。


#### 5.5 CopyOut：将结果写回 GM

CopyOut 从输出队列取出已经完成计算的 LocalTensor，使用与 CopyIn 相同的核内 Tile 偏移写回 <code>zGm_</code>，最后释放输出槽位。


In [ ]:
show_between(
    SOURCE,
    '    __aicore__ inline void CopyOut',
    'private:',
)


<strong>观察结果：</strong> 同一个 <code>progress</code> 在 CopyIn 和 CopyOut 中产生相同的 <code>offset</code>，因此输入 Tile 与输出 Tile 一一对应。<code>FreeTensor</code> 位于 <code>DataCopy</code> 之后，避免输出仍在搬运时提前复用槽位。

至此，一个 Tile 的完整生命周期为：

<pre><code>GM x/y → Alloc → CopyIn → EnQue
       → DeQue → Add → EnQue z → Free x/y
       → DeQue z → CopyOut → Free z → GM z</code></pre>


#### 5.6 Kernel 入口：连接 Host 参数与算子类

<code>__global__ __aicore__</code> 函数是 Host 发射的 Device 入口。它声明本算子使用 Vector Core，创建算子对象，然后把 GM 地址和 Tiling 参数交给 <code>Init</code> 与 <code>Process</code>。


In [ ]:
show_between(
    SOURCE,
    '__global__ __aicore__ void vector_add(',
    '// ---------------- Host runner',
)


<strong>完成检查：</strong> 能够从 Kernel 入口开始，指出 <code>tiling.totalLength</code> 和 <code>tiling.tileCount</code> 最终在哪些公式中被使用；能够说明 <code>blockDim</code> 不在 Tiling 结构体内，而是在 Host 发射语法中指定。


### 6. 理解 Host 侧启动流程

本实验的重点是 Vector Kernel，因此 Host 只沿主流程阅读，不展开每个 ACL 资源管理函数。AscendCL（代码中以 <code>acl*</code> 接口出现，简称 ACL）是 Host 侧管理设备、内存与任务执行的运行时接口；H2D 和 D2H 分别表示 Host→Device 与 Device→Host 数据拷贝；CPU Golden 是在 CPU 上计算的参考结果，用于逐元素核对 NPU 输出。完整实现可在 [src/demo/vector_add.asc](src/demo/vector_add.asc) 中查看。

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>步骤</th><th style='text-align: left;'>Host 操作</th><th style='text-align: left;'>目的</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'>1</td><td style='text-align: left;'>解析参数并检查 Shape</td><td style='text-align: left;'>在发射前拒绝无法完整切分或未对齐的参数</td></tr>
    <tr><td style='text-align: left;'>2</td><td style='text-align: left;'>准备输入、初始化 ACL、分配 Device 内存</td><td style='text-align: left;'>建立 Host 与 Device 运行环境</td></tr>
    <tr><td style='text-align: left;'>3</td><td style='text-align: left;'>H2D 拷贝</td><td style='text-align: left;'>将 <code>x/y</code> 放入 GM</td></tr>
    <tr><td style='text-align: left;'>4</td><td style='text-align: left;'>组装 Tiling、预热、发射并同步</td><td style='text-align: left;'>在指定 <code>blockDim</code> 下执行 Kernel</td></tr>
    <tr><td style='text-align: left;'>5</td><td style='text-align: left;'>D2H 拷贝与 CPU Golden 对比</td><td style='text-align: left;'>验证每一个输出元素</td></tr>
    <tr><td style='text-align: left;'>6</td><td style='text-align: left;'>释放资源并打印指标</td><td style='text-align: left;'>形成可复核的实验记录</td></tr>
  </tbody>
</table>


#### 6.1 在 Kernel 发射前检查输入

运行下面的单元查看 <code>ValidateShape</code>。阅读时将每个 <code>return false</code> 与第 1 节的三个参数条件对应起来。


In [ ]:
show_between(
    SOURCE,
    'bool ValidateShape',
    'void FillInput',
)


这三道检查按依赖顺序执行：只有 <code>N</code> 能被 <code>blockDim</code> 整除，<code>blockLength</code> 才是整数；只有 <code>blockLength</code> 能被 <code>tileCount</code> 整除，<code>tileLength</code> 才能完整覆盖当前核；最后再检查单次 <code>DataCopy</code> 的字节数。

把检查放在 Host 侧，可以在分配 Device 内存和发射 Kernel 之前给出明确错误，避免越界、未覆盖元素或不满足搬运约束。


#### 6.2 组装 Tiling、发射并校验

下面只定位 Host 主流程中的关键语句，不输出整段 ACL 资源管理代码。请依次确认 Tiling 组装、Kernel 发射、Stream 同步、CPU Golden 对比和计时口径。


In [ ]:
show_matches(
    SOURCE,
    [
        'const VectorAddTilingData tiling',
        'vector_add<<<options.blockDim',
        'aclrtSynchronizeStream',
        'aclrtMemcpy(output.data()',
        'const float error =',
        'timing_scope=launch_plus_sync',
    ],
)


输出字段 <code>avg_kernel_us</code> 的名称便于实验记录，但必须结合 <code>timing_scope=launch_plus_sync</code> 理解：

- Host 在预热后开始计时；
- 计时区间包含多次 Kernel 发射和随后的一次 Stream 同步；
- 总时间除以 <code>iterations</code> 得到 <code>avg_kernel_us</code>；
- H2D 与 D2H 不在计时区间内；
- 它是 Host 侧 <code>launch_plus_sync</code> 平均值，不是设备侧纯 Kernel Task Duration。若需要纯设备时间，应使用 <code>msprof</code> 的 Task Duration。

<code>effective_gb_s</code> 也基于同一 Host 侧时间口径计算，因此记录性能数据时应同时保留 <code>timing_scope</code>。


### 7. 配置 CMake

演示工程同时包含 ASC Device 代码和 C++ Host 代码。CMake 需要找到 ASC 编译规则，创建可执行目标，并为 ASC 源指定当前实验设备的架构。运行下面的单元查看完整但很短的构建文件。


In [ ]:
print(CMAKE_FILE.read_text(encoding='utf-8'))


阅读结果应包含：

- <code>find_package(ASC REQUIRED)</code>：接入 Ascend C 编译规则；
- <code>project(... LANGUAGES ASC CXX)</code>：声明 ASC 与 C++ 两种语言；
- <code>add_executable(vector_add vector_add.asc)</code>：生成实验程序；
- <code>--npu-arch=dav-2201</code>：本课程 A2/Ascend 910B3 的已验证基线。

若 CANNLab 分配的硬件不同，应先记录实际设备与 CANN 版本，再按环境要求覆盖 <code>NPU_ARCH</code>，不要直接猜测架构值。


### 8. 编译并运行默认实验

下面的单元会在 <code>/tmp</code> 下创建独立构建目录，不修改课程源码。它先执行干净配置和编译，再以默认教学参数运行一次。

<strong>运行前确认：</strong> 当前 Notebook 使用 CANNLab 的 CANN Python 内核，ASC 工具链已加载，设备可用。  
<strong>成功判据：</strong> 编译和程序退出码均为 0，指标行包含 <code>buffer_num=1</code>、<code>correctness=PASS</code>，且 <code>max_abs_error ≤ 1e-6</code>。在已验证的默认配置中，最大误差为 0。


In [ ]:
if WORK.exists():
    shutil.rmtree(WORK)
build = WORK / 'build'
build.mkdir(parents=True)
project = CHAPTER / 'src/demo'

subprocess.run(['cmake', '-S', str(project), '-B', str(build)], check=True)
subprocess.run(['cmake', '--build', str(build), '-j'], check=True)
exe = next(p for p in build.rglob('vector_add') if p.is_file() and p.name == 'vector_add')

result = subprocess.run(
    [
        str(exe),
        '--length', '16384',
        '--block-dim', '8',
        '--tile-count', '8',
        '--seed', '1',
    ],
    text=True,
    capture_output=True,
)
print(result.stdout, end='')
if result.stderr:
    print(result.stderr, end='')
assert result.returncode == 0
assert 'correctness=PASS' in result.stdout
assert 'buffer_num=1' in result.stdout
assert 'timing_scope=launch_plus_sync' in result.stdout


代表性指标行如下，时间和带宽会随设备负载变化：

<pre><code>METRIC buffer_num=1 compute_repeat=1 length=16384 block_dim=8 tile_count=8 tile_length=256 queue_bytes=3072 correctness=PASS max_abs_error=0.000000 timing_scope=launch_plus_sync avg_kernel_us=&lt;实测值&gt; effective_gb_s=&lt;实测值&gt;</code></pre>

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>字段</th><th style='text-align: left;'>如何解释</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>buffer_num=1</code></td><td style='text-align: left;'><code>InitBuffer</code> 为每个 TQue 分配 1 个物理 Buffer；queue depth 固定为 1</td></tr>
    <tr><td style='text-align: left;'><code>tile_count=8</code></td><td style='text-align: left;'>每核处理 8 个真实 GM Tile</td></tr>
    <tr><td style='text-align: left;'><code>tile_length=256</code></td><td style='text-align: left;'>每个 Tile 包含 256 个 <code>float</code> 元素</td></tr>
    <tr><td style='text-align: left;'><code>queue_bytes=3072</code></td><td style='text-align: left;'>单个逻辑 Block（每核实例）三个队列的 Local Memory 槽位总量</td></tr>
    <tr><td style='text-align: left;'><code>correctness=PASS</code></td><td style='text-align: left;'>所有元素通过有限数与最大绝对误差检查</td></tr>
    <tr><td style='text-align: left;'><code>max_abs_error</code></td><td style='text-align: left;'>通过条件为不超过 <code>1e-6</code>；已验证的默认配置结果为 0</td></tr>
    <tr><td style='text-align: left;'><code>avg_kernel_us</code></td><td style='text-align: left;'>Host 侧 <code>launch_plus_sync</code> 平均时间</td></tr>
  </tbody>
</table>

不要只记录耗时。一次完整的运行记录应同时包含 Shape、切分参数、精度结果和计时口径，性能数据才可复核。


### 9. 观察非法参数如何被拒绝

运行下面的单元，分别触发“不能等分到 Block”“不能等分到 Tile”和“单次搬运未按 32 Byte 对齐”三类错误。三个用例都应在 Kernel 发射前以退出码 2 结束，因此不会访问设备数据。


In [ ]:
invalid_cases = [
    (
        '分核不完整',
        ['--length', '16385', '--block-dim', '8', '--tile-count', '8'],
        'length must be divisible by blockDim',
    ),
    (
        '核内分 Tile 不完整',
        ['--length', '16384', '--block-dim', '8', '--tile-count', '7'],
        'block length must be divisible by tileCount',
    ),
    (
        '搬运长度未对齐',
        ['--length', '80', '--block-dim', '1', '--tile-count', '8'],
        'each DataCopy length must be 32-byte aligned',
    ),
]

for label, arguments, expected_message in invalid_cases:
    failed = subprocess.run(
        [str(exe), *arguments],
        text=True,
        capture_output=True,
    )
    first_line = (failed.stderr or failed.stdout).strip().splitlines()[0]
    print(f'{label}: returncode={failed.returncode}, message={first_line}')
    assert failed.returncode == 2
    assert expected_message in failed.stderr


<strong>预期现象：</strong> 三行结果的 <code>returncode</code> 都为 2，并依次出现对应的 Shape 提示。若程序进入 ACL 初始化或 Kernel 发射阶段，说明 Host 参数检查没有按预期生效。

这一步说明“参数不受支持”与“计算结果错误”是两类不同问题：前者应由 Host 尽早拒绝，后者才需要查看 Kernel 数据通路和精度比较。


### 10. 结果解释与常见问题

遇到失败时，从最早出现的错误开始排查，不要先修改 Kernel。

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>现象</th><th style='text-align: left;'>优先检查</th><th style='text-align: left;'>处理建议</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'>CMake 找不到 ASC</td><td style='text-align: left;'>CANN 环境是否加载、Notebook 内核是否正确</td><td style='text-align: left;'>回到 01.02 重新检查环境变量和编译工具路径</td></tr>
    <tr><td style='text-align: left;'><code>length must be divisible...</code></td><td style='text-align: left;'><code>N % blockDim</code></td><td style='text-align: left;'>调整长度或 Block 数，使每核元素数为整数</td></tr>
    <tr><td style='text-align: left;'><code>block length must be divisible...</code></td><td style='text-align: left;'><code>blockLength % tileCount</code></td><td style='text-align: left;'>重新选择每核 Tile 数</td></tr>
    <tr><td style='text-align: left;'><code>32-byte aligned</code></td><td style='text-align: left;'><code>tileLength × 4</code></td><td style='text-align: left;'>使 <code>float32</code> Tile 元素数为 8 的整数倍</td></tr>
    <tr><td style='text-align: left;'>ACL 初始化或分配失败</td><td style='text-align: left;'>设备是否可见、资源是否被占用</td><td style='text-align: left;'>先确认 <code>npu-smi info</code>，必要时更换空闲设备</td></tr>
    <tr><td style='text-align: left;'><code>correctness=FAIL</code></td><td style='text-align: left;'>Block 偏移、Tile 偏移、队列生命周期、Add 长度</td><td style='text-align: left;'>用一个固定 Block/Tile 手算 GM 区间，从首个错误位置回溯</td></tr>
    <tr><td style='text-align: left;'><code>max_abs_error</code> 为 NaN 或 Inf</td><td style='text-align: left;'>是否写回全部元素、是否使用了未初始化 LocalTensor</td><td style='text-align: left;'>检查 CopyOut 与输出队列的出队顺序</td></tr>
    <tr><td style='text-align: left;'>时间波动较大</td><td style='text-align: left;'>预热、迭代次数、设备负载与计时口径</td><td style='text-align: left;'>保持参数一致，多轮运行并记录中位数；不要把 Host 时间当作 Task Duration</td></tr>
  </tbody>
</table>


## 课后练习

### 11. 巩固练习

请先独立完成并把推导、命令和观察结果写入实验记录。

1. 对 <code>N=32768</code>、<code>blockDim=4</code>、<code>tileCount=16</code>，推导 <code>blockLength</code>、<code>tileLength</code> 和 <code>tileBytes</code>，并判断三个参数条件是否全部满足。
2. 使用上述合法配置运行程序，记录并解释 <code>tile_length</code>、<code>queue_bytes</code>、<code>correctness</code>、<code>max_abs_error</code> 和 <code>timing_scope</code>。不要把一次运行的耗时直接当作性能结论。
3. 把长度改为 <code>32769</code>，记录退出码和第一条错误信息，说明为什么应在 Kernel 发射前拒绝。
4. 假设 Init 遗漏 <code>GetBlockIdx()</code> 对 GM 地址的偏移，分析哪些全局区间会被重复写入、哪些区间不会被覆盖。
5. 画出一个输入 Tile 和一个输出 Tile 的队列状态变化，解释为什么 <code>EnQue</code> 必须早于下一阶段的 <code>DeQue</code>，以及为什么消费后必须 <code>FreeTensor</code>。
6. 用自己的话区分 <code>tileCount=8</code> 与 <code>buffer_num=1</code>：二者分别控制什么，哪一个决定每核循环次数？

7. 完成下面的 <code>run_case.py</code>：组装第 1 题的启动参数，并同时检查 Shape 字段与 <code>correctness=PASS</code>。练习脚本与教程源码分离，不修改 <code>src/demo</code>。

独立完成后，可使用下方开关查看参考讲解。参考文件位于 <code>answer/01.03_vector_add_operator/answers.md</code>，默认运行 Notebook 不会显示答案。


#### 编程实践：自动运行并检查一组 VectorAdd 参数

下面先在当前用户专属的临时课程根目录中创建练习目录，随后通过 <code>%%writefile</code> 提供待填写文件。该目录按系统用户标识隔离（支持 UID 的系统使用 UID，否则使用登录用户名），不会与其他用户冲突。补全两个 TODO 后，脚本应以退出码 0 结束并打印 <code>PRACTICE PASS</code>。


In [ ]:
PRACTICE = USER_TEMP_ROOT / '01_vector_add_operator_practice'
PRACTICE.mkdir(parents=True, exist_ok=True)
PRACTICE_FILE = PRACTICE / 'run_case.py'
print('practice file:', PRACTICE_FILE)


In [ ]:
%%writefile {PRACTICE_FILE}
import subprocess
import sys

def build_command(executable):
    # TODO 1：返回 N=32768、blockDim=4、tileCount=16 的完整命令列表。
    return [executable]

def metric_pass(text):
    # TODO 2：同时检查 Shape、tile_length=512 与 correctness=PASS。
    return False

result = subprocess.run(build_command(sys.argv[1]), text=True, capture_output=True)
print(result.stdout)
passed = result.returncode == 0 and metric_pass(result.stdout)
print('PRACTICE PASS' if passed else 'PRACTICE TODO')
raise SystemExit(0 if passed else 1)


In [ ]:
practice_result = subprocess.run(
    [sys.executable, str(PRACTICE_FILE), str(build / 'vector_add')],
    text=True,
    capture_output=True,
)
print(practice_result.stdout)
print('practice exit code:', practice_result.returncode)


#### 独立完成后查看参考答案

下面的单元默认设置 <code>SHOW_ANSWER = False</code>。请先保存自己的推导和运行记录；确认独立完成后，再把开关改为 <code>True</code> 并单独运行该单元。


In [ ]:
SHOW_ANSWER = False

if SHOW_ANSWER:
    subprocess.run(['cat', str(CHAPTER / 'answer/01.03_vector_add_operator/answers.md')], check=True)
    subprocess.run(['cat', str(CHAPTER / 'answer/01.03_vector_add_operator/run_case.py')], check=True)
else:
    print('参考答案保持隐藏；独立完成后可将 SHOW_ANSWER 改为 True。')


查看答案后，请回到自己的实验记录，补充推导依据、实际输出和错误定位过程，不要只抄写最终结论。


## 本节小结与完成检查

本节把一维数组切分落实为一个可运行的 Ascend C VectorAdd：Host 先验证 Shape 并准备 Tiling，Kernel 在 Init 中绑定每核 GM 区间，在 Process 中按真实 Tile 数循环，每个 Tile 依次完成 CopyIn、Compute 和 CopyOut。

进入下一节前，请确认你已经完成：

- 能计算任意 Block、Tile 对应的全局半开区间；
- 能说明 <code>GlobalTensor</code>、<code>LocalTensor</code>、<code>TPipe</code> 和 <code>TQue</code> 的作用；
- 能按正确顺序写出三个阶段的队列生命周期；
- 已完成默认合法配置并看到 <code>correctness=PASS</code>；
- 已观察三类非法参数在 Host 侧被拒绝；
- 能解释 <code>tileCount</code> 与队列槽位数相互独立；
- 能说明 <code>avg_kernel_us</code> 是 Host 侧 <code>launch_plus_sync</code>，不是设备侧 Task Duration。


完成后继续学习 [01.04 Tiling 可视化交互实验](01.04_tiling_visualization.ipynb)。
